---
## Crop pharyngeal bulb ROI using FRCNN tracks
Crops a folder of videos into 250x250p regions of interest centered on the tracked pharyngeal bulb. **Make sure that you have run `EZ_FRCNN.ipynb` to track the pharyngeal bulb first!**

In [ ]:
### Import libraries + define paths
from library import *

VID_DIR = './videos/'  # Directory containing videos to be analyzed
OUT_DIR = './outputs/' # Directory where all code outputs (FRCNN tracks, plots) are stored

### Get names of all videos within the folder
video_type  = '.wmv' # CHANGE FOR YOUR DATA
video_paths = glob.glob(VID_DIR + '*' + video_type)  
videos      = [os.path.splitext(os.path.basename(v))[0] for v in video_paths]

In [ ]:
### Set parameters
ROI_width = 125   # resulting video will be 2*roi_width x 2*roi_width
thresh    = 250   # threshold for filtering ROI coordinates
sig       = 7     # sigma used for smoothing the ROI centers

### Dynamically crop all videos in recording
for video in videos:
    ### Load data
    fullName = video + video_type
    vidPath  = VID_DIR + fullName

    bboxName = OUT_DIR + '/frcnn_bulb/' + video + '_boxes.csv'
    boxes    = pd.read_csv(bboxName)
    boxes    = boxes.to_numpy()

    ### Load video
    vid        = cv2.VideoCapture(vidPath)
    width      = int(vid.get(cv2.CAP_PROP_FRAME_WIDTH))
    height     = int(vid.get(cv2.CAP_PROP_FRAME_HEIGHT))
    fps        = vid.get(cv2.CAP_PROP_FPS)
    tot_frames = int(vid.get(cv2.CAP_PROP_FRAME_COUNT)) 

    # Create name for output video
    fileName = VID_DIR + '/cropped/' + video + '_cropped' + video_type
    vwriter  = cv2.VideoWriter(fileName,
                              cv2.VideoWriter_fourcc('M','J','P','G'),
                              fps, (2*ROI_width, 2*ROI_width))

    # Get unfiltered ROI coordinates
    ROI_coords = np.zeros((tot_frames,2)) # array to store tracked bulb centers
    for f in range(0,tot_frames-1):
        # Grab the center of each tracked pharyngeal bulb, then process after
        ROI = boxes[f].astype('int')
        ROI_coords[f] = [np.mean([ROI[0], ROI[2]]).astype('int'), np.mean([ROI[1], ROI[3]]).astype('int')]
    
    # Filter and smooth ROI coordinates
    ROI_coords_i, ROI_coords_s = filter_ROIs(ROI_coords, threshold=thresh, sigma=sig, max_values=(width,height))

    ## SECOND LOOP: crop the video frame by frame
    f = 1
    success = True
    while success and f < tot_frames:
        # Read next frame
        success, frame = vid.read()

        if success and f < tot_frames:
            print("frame: ", f-1)

            # Crop out the ROI from the current frame
            ROI = getROI(ROI_width, ROI_coords_s[f], frame, width, height)

            # Save roi frame to video
            vwriter.write(ROI)

        # Move to next frame
        f += 1
        clear_output(wait=True)

    vwriter.release()
    clear_output(wait=True)
    print("Saved to ", fileName)